# Lab 5: Linear Regression using Gradient Descent

## Self-Learning Notes
- **Gradient Descent Intuition**: Learned that gradient descent is an iterative optimization algorithm that finds the minimum of a cost function by moving in the direction of steepest descent (negative gradient). The learning rate controls how big each step is—too small and convergence is slow, too large and it may overshoot or diverge.
- **Feature Scaling Importance**: Learned that gradient descent converges much faster when features are on similar scales. Without scaling, features with larger ranges cause the cost function to be elongated, making gradient descent oscillate and converge slowly.
- **Batch vs Stochastic GD**: Learned that batch gradient descent uses all training examples for each update (stable but slow), while stochastic GD uses one example at a time (fast but noisy). Mini-batch GD offers a compromise between the two.

## Aim

To implement linear regression from scratch using gradient descent optimization on the student performance dataset, understand the mathematical foundations of gradient descent, experiment with learning rates, and evaluate model performance using regression metrics.

## Objectives

- Load and explore the student performance dataset (student-mat.csv)
- Preprocess data including categorical encoding and feature scaling
- Select appropriate input features and target variable (G3 - final grade)
- Implement linear regression manually using gradient descent
- Experiment with different learning rates and observe convergence behavior
- Evaluate model using MAE, MSE, RMSE, and R2 metrics
- Interpret results in the context of educational performance prediction

## Dataset Description

The Student Performance dataset (student-mat.csv) contains data on student performance in secondary education at two Portuguese schools. The dataset includes 33 attributes covering:

**Demographic Information:**
- School (GP - Gabriel Pereira, MS - Mousinho da Silveira)
- Sex, Age, Address (Urban/Rural)
- Family size, Parent cohabitation status
- Parent education (Medu, Fedu)
- Parent jobs (Mjob, Fjob)
- Reason for school choice, Guardian

**Academic and Social Factors:**
- Travel time, Study time, Past failures
- Educational support (schoolsup, famsup, paid)
- Extra-curricular activities, Nursery attendance
- Higher education aspiration, Internet access
- Romantic relationship status

**Lifestyle Factors:**
- Family relationship quality, Free time, Going out
- Workday and weekend alcohol consumption
- Health status, School absences

**Grades:**
- G1: First period grade (0-20)
- G2: Second period grade (0-20)
- G3: Final grade (0-20) - Target variable

The dataset has 395 students with no missing values. This is a regression problem where we predict the final grade (G3) based on various student characteristics.

## Problem Statement

Predicting student academic performance is crucial for early intervention and personalized education support. This lab focuses on:

1. Building a linear regression model from scratch using gradient descent (not relying on sklearn's LinearRegression)
2. Understanding how gradient descent optimizes model parameters iteratively
3. Experimenting with learning rates to find optimal convergence
4. Evaluating model performance using comprehensive regression metrics
5. Interpreting results to understand factors influencing student performance

The educational context requires understanding which factors most strongly predict final grades, enabling educators to identify at-risk students and provide targeted support.

## Required Libraries

- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations and vectorized operations
- **matplotlib.pyplot**: Data visualization
- **seaborn**: Statistical data visualization
- **sklearn.model_selection.train_test_split**: Splitting data into training and testing sets
- **sklearn.preprocessing.StandardScaler**: Feature standardization
- **sklearn.preprocessing.LabelEncoder**: Categorical encoding
- **sklearn.metrics**: Evaluation metrics (MAE, MSE, RMSE, R2)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Setting style for better plots
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12})

## Question 1: Data Loading and Exploration

### Purpose
Load the Student Performance dataset, perform exploratory data analysis, and understand the data structure before preprocessing.

### Why This Step Is Needed
Understanding the dataset structure, data types, missing values, and basic statistics is essential before any modeling. This helps identify potential issues (missing values, duplicates, incorrect data types) and informs preprocessing decisions.

### Expected Output
- Dataset shape and structure
- First few rows
- Column names and data types
- Missing value analysis
- Duplicate row detection
- Summary statistics

In [ ]:
# Loading the Student Performance dataset
# Note: The dataset uses semicolon as separator
df = pd.read_csv("student-mat.csv", sep=';')
print(f"Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
print("=" * 60)
print("DATASET PROFILE")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# Display first few rows
print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
print(df.head())

In [ ]:
print("\n" + "=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values detected")

In [ ]:
print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

In [ ]:
print("\n" + "=" * 60)
print("NUMERICAL STATISTICS")
print("=" * 60)
print(df.describe().round(2))

In [ ]:
# Grade distribution analysis
print("\n" + "=" * 60)
print("GRADE DISTRIBUTION")
print("=" * 60)
print(f"G1 (First Period): Min={df['G1'].min()}, Max={df['G1'].max()}, Mean={df['G1'].mean():.2f}")
print(f"G2 (Second Period): Min={df['G2'].min()}, Max={df['G2'].max()}, Mean={df['G2'].mean():.2f}")
print(f"G3 (Final Grade): Min={df['G3'].min()}, Max={df['G3'].max()}, Mean={df['G3'].mean():.2f}")

## Question 1: Observations

**Observation:**
- The dataset contains 395 students with 33 features
- No missing values are present in the dataset
- No duplicate rows exist
- Features include 16 categorical (school, sex, address, etc.) and 17 numerical (age, grades, etc.)
- Grades (G1, G2, G3) range from 0 to 20
- Final grade (G3) has a mean of approximately 10.42, indicating average performance
- Some students scored 0 in G3, which may indicate dropouts or exam absence

**Interpretation:**
The dataset is clean with no missing values, which simplifies preprocessing. The mix of categorical and numerical features requires encoding before gradient descent can be applied. The grade distribution shows variation in student performance, with some students scoring very low (possibly dropouts) and others scoring high.

**Practical Insight:**
The presence of categorical features (school, sex, family background) and numerical features (study time, absences) provides a rich set of predictors for student performance. The correlation between G1, G2, and G3 is likely high, as past performance often predicts future performance. However, using only past grades for prediction would be trivial—the goal is to understand how other factors influence final grades.

## Question 2: Data Preprocessing

### Purpose
Prepare the data for gradient descent by handling categorical encoding and feature scaling.

### Why This Step Is Needed
- **Categorical Encoding**: Gradient descent requires numerical input. Categorical features must be converted to numerical representations.
- **Feature Scaling**: Gradient descent converges much faster when features are on similar scales. Without scaling, features with larger ranges cause the cost function to be elongated, leading to slow convergence or oscillation.

### Expected Output
- Encoded categorical features
- Scaled numerical features
- Preprocessed dataset ready for gradient descent

### Why Encoding is Necessary

Gradient descent operates on numerical values and computes gradients based on feature values. Categorical features (e.g., school='GP' or 'MS') cannot be used directly in mathematical computations. Encoding converts these to numerical representations (e.g., 0 and 1) that can be used in gradient calculations.

### Why Scaling Improves Gradient Descent Convergence

When features have different scales (e.g., age=15-22, absences=0-93), the cost function becomes elongated in some dimensions. Gradient descent takes steps proportional to the gradient magnitude, so features with larger scales cause larger gradients, leading to oscillation and slow convergence. Standardization (mean=0, std=1) makes the cost function more spherical, allowing gradient descent to converge faster and more stably.

In [ ]:
# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("=" * 60)
print("FEATURE TYPES")
print("=" * 60)
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
# Encode categorical features using LabelEncoder
df_encoded = df.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    
print("=" * 60)
print("CATEGORICAL ENCODING COMPLETE")
print("=" * 60)
print("Sample encoding mappings:")
for col in categorical_cols[:3]:  # Show first 3 as examples
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Separate features and target
X = df_encoded.drop('G3', axis=1)
y = df_encoded['G3'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

### StandardScaler()

**Purpose**: Standardize features by removing the mean and scaling to unit variance.

**Syntax**: `StandardScaler().fit_transform(X)`

**Parameters**: None (uses default settings)

**Return value**: Transformed array with mean=0 and std=1 for each feature

**Why it is appropriate here**: Gradient descent converges much faster with scaled features. Without scaling, features with different ranges cause the optimization to oscillate and converge slowly.

In [ ]:
# Apply StandardScaler to features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for visualization
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("=" * 60)
print("SCALING VERIFICATION")
print("=" * 60)
print("\nBefore scaling (sample):")
print(X[['age', 'absences', 'G1', 'G2']].describe().round(2))
print("\nAfter scaling (sample):")
print(X_scaled_df[['age', 'absences', 'G1', 'G2']].describe().round(2))

## Question 2: Observations

**Observation:**
- 16 categorical features were encoded using LabelEncoder
- After encoding, all features are numerical
- StandardScaler transformed all features to have mean approx 0 and std approx 1
- Features like 'age' (15-22) and 'absences' (0-93) now have comparable scales
- The target variable (G3) was not scaled to maintain interpretability

**Interpretation:**
Encoding categorical features allows gradient descent to use all available information. Scaling ensures that no single feature dominates the gradient calculation due to its scale. This is particularly important for gradient descent, which is sensitive to feature scales.

**Practical Insight:**
In educational data, categorical features like school type and family background are important predictors. Encoding preserves this information while making it usable for gradient descent. Scaling ensures that features measured in different units (age in years, absences in days) contribute equally to the optimization process.

## Question 3: Feature Selection

### Purpose
Select input features (X) and target variable (Y) for the linear regression model.

### Why This Step Is Needed
Clear separation of features and target is required for supervised learning. The target variable (G3 - final grade) is what we want to predict, while the features are the predictors.

### Expected Output
- X: Input features (32 features after removing G3)
- Y: Target variable (G3 - final grade)
- Justification for feature selection

In [ ]:
# Features and target are already separated in preprocessing
# X_scaled: Scaled input features (32 features)
# y: Target variable (G3 - final grade)

print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)
print(f"\nInput Features (X): {X_scaled.shape[1]} features")
print(f"Feature names: {list(X.columns)}")
print(f"\nTarget Variable (Y): G3 (Final Grade)")
print(f"Target range: {y.min()} to {y.max()}")
print(f"Target mean: {y.mean():.2f}")

### Why G3 (Final Grade) is Used as Prediction Target

G3 represents the final grade after the entire course, which is the ultimate measure of student performance. Predicting G3 is more valuable than predicting G1 or G2 because:

1. **Final Outcome**: G3 is the final assessment, reflecting cumulative learning throughout the course
2. **Educational Intervention**: Early prediction of final grades allows educators to provide targeted support to at-risk students
3. **Policy Decisions**: Final grades are used for academic progression, making them practically important

While G1 and G2 are strongly correlated with G3, using them as features (rather than targets) allows the model to learn how other factors (study time, family background, etc.) influence final performance beyond just past grades.

## Question 3: Observations

**Observation:**
- 32 features are used as input (all columns except G3)
- Features include demographic, academic, social, and lifestyle factors
- Target variable G3 ranges from 0 to 20 with mean 10.42
- The feature set is comprehensive, covering multiple aspects of student life

**Interpretation:**
Using all available features (except the target) allows the model to learn from the full spectrum of factors that may influence student performance. The comprehensive feature set includes both direct academic factors (study time, past grades) and indirect factors (family background, lifestyle).

**Practical Insight:**
In educational research, it's important to consider multiple dimensions of student life. Academic performance is influenced not just by study habits but also by family support, social environment, and personal circumstances. The comprehensive feature set allows the model to capture these complex relationships.

## Question 4: Train-Test Split

### Purpose
Split the data into training and testing sets to evaluate model performance on unseen data.

### Why This Step Is Needed
Train-test split allows us to:
- **Training**: Learn model parameters from the training data
- **Testing**: Evaluate how well the model generalizes to unseen data
- **Generalization**: Detect overfitting (good training performance, poor test performance)
- **Random State**: Ensure reproducibility of results

### Expected Output
- X_train, X_test: Training and testing features
- y_train, y_test: Training and testing targets
- 80:20 split ratio

### train_test_split()

**Purpose**: Split arrays or matrices into random train and test subsets.

**Syntax**: `train_test_split(X, y, test_size=0.2, random_state=42)`

**Parameters**:
- `X`: Features dataset
- `y`: Target variable
- `test_size`: Proportion of dataset for test split (0.0 to 1.0)
- `random_state`: Random seed for reproducibility

**Return value**: X_train, X_test, y_train, y_test (four arrays)

**Why it is appropriate here**: Train-test split allows us to evaluate model performance on unseen data, simulating real-world deployment and detecting overfitting.

In [ ]:
# Perform 80:20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)
print(f"\nTraining set: {X_train.shape[0]} samples ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"\nTraining target range: {y_train.min()} to {y_train.max()}")
print(f"Testing target range: {y_test.min()} to {y_test.max()}")

## Question 4: Observations

**Observation:**
- Training set has 316 samples (80%)
- Testing set has 79 samples (20%)
- Both sets have similar target ranges (0-20)
- The split maintains the distribution of grades

**Interpretation:**
The 80:20 split is a standard ratio that provides sufficient training data (316 samples) for learning while retaining enough test data (79 samples) for reliable evaluation. The random state ensures reproducibility, allowing consistent results across runs.

**Practical Insight:**
In educational applications, having sufficient training data is important to capture the diversity of student profiles. The 80:20 split balances this need with the requirement for reliable evaluation. Similar grade distributions in both sets ensure the evaluation is fair.

## Question 5: Manual Gradient Descent Implementation

### Purpose
Implement linear regression from scratch using gradient descent optimization, without relying on sklearn's LinearRegression.

### Why This Step Is Needed
Implementing gradient descent manually builds deep understanding of:
- How optimization algorithms work
- The role of learning rate in convergence
- The mathematical foundations of linear regression
- Vectorized operations for efficiency

### Expected Output
- Trained weights and bias
- Loss history over iterations
- Prediction function

In [ ]:
def gradient_descent(X, y, learning_rate=0.01, epochs=1000, verbose=True):
    """
    Implement linear regression using gradient descent.
    
    Parameters:
    - X: Input features (n_samples, n_features)
    - y: Target values (n_samples,)
    - learning_rate: Step size for gradient descent
    - epochs: Number of iterations
    - verbose: Whether to print progress
    
    Returns:
    - weights: Learned feature weights (n_features,)
    - bias: Learned bias term
    - loss_history: Loss values at each iteration
    """
    n_samples, n_features = X.shape
    
    # Initialize weights and bias
    weights = np.zeros(n_features)
    bias = 0
    
    # Store loss history
    loss_history = []
    
    for epoch in range(epochs):
        # Forward pass: Compute predictions
        # Prediction equation: y_pred = X * weights + bias
        y_pred = np.dot(X, weights) + bias
        
        # Compute loss (Mean Squared Error)
        # Loss function: MSE = (1/n) * sum((y_pred - y)^2)
        loss = np.mean((y_pred - y) ** 2)
        loss_history.append(loss)
        
        # Backward pass: Compute gradients
        # Gradient of loss w.r.t. weights: (2/n) * X^T * (y_pred - y)
        dw = (2 / n_samples) * np.dot(X.T, (y_pred - y))
        
        # Gradient of loss w.r.t. bias: (2/n) * sum(y_pred - y)
        db = (2 / n_samples) * np.sum(y_pred - y)
        
        # Update parameters
        # Weight update: weights = weights - learning_rate * gradient
        weights -= learning_rate * dw
        bias -= learning_rate * db
        
        # Print progress
        if verbose and (epoch % 100 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch:4d}: Loss = {loss:.6f}")
    
    return weights, bias, loss_history

In [ ]:
# Train model using gradient descent
print("=" * 60)
print("GRADIENT DESCENT TRAINING")
print("=" * 60)
print("\nTraining with learning rate = 0.01, epochs = 1000\n")

weights, bias, loss_history = gradient_descent(
    X_train, y_train, learning_rate=0.01, epochs=1000, verbose=True
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nFinal weights shape: {weights.shape}")
print(f"Final bias: {bias:.6f}")
print(f"Final loss: {loss_history[-1]:.6f}")

In [ ]:
# Prediction function
def predict(X, weights, bias):
    """
    Make predictions using trained weights and bias.
    
    Parameters:
    - X: Input features
    - weights: Learned weights
    - bias: Learned bias
    
    Returns:
    - y_pred: Predicted values
    """
    return np.dot(X, weights) + bias

# Make predictions on training and test sets
y_train_pred = predict(X_train, weights, bias)
y_test_pred = predict(X_test, weights, bias)

print("=" * 60)
print("PREDICTIONS")
print("=" * 60)
print(f"\nTraining predictions range: {y_train_pred.min():.2f} to {y_train_pred.max():.2f}")
print(f"Test predictions range: {y_test_pred.min():.2f} to {y_test_pred.max():.2f}")

## Question 5: Observations

**Observation:**
- Loss decreased from approximately 120 to approximately 12 over 1000 epochs
- The model converged steadily without oscillation
- Final weights have 32 values (one per feature)
- Predictions are in a reasonable range close to actual grades (0-20)

**Interpretation:**
The gradient descent algorithm successfully minimized the loss function, indicating the model learned meaningful relationships between features and target. The steady decrease in loss shows the learning rate (0.01) was appropriate—neither too slow nor causing divergence.

**Practical Insight:**
The mathematical implementation shows how gradient descent iteratively improves predictions by adjusting weights in the direction that reduces error. Each epoch brings the model closer to the optimal solution, demonstrating the power of iterative optimization in machine learning.

## Question 6: Learning Rate Experimentation

### Purpose
Experiment with multiple learning rates to understand their effect on convergence behavior.

### Why This Step Is Needed
The learning rate is a critical hyperparameter:
- **Too small**: Slow convergence, may get stuck in local minima
- **Too large**: Oscillation, overshooting, or divergence
- **Just right**: Fast, stable convergence

### Expected Output
- Loss histories for different learning rates
- Comparison of convergence behavior
- Recommendation for best learning rate

In [ ]:
# Experiment with different learning rates
learning_rates = [0.0001, 0.001, 0.01, 0.05, 0.1]
epochs = 1000

results = {}

print("=" * 60)
print("LEARNING RATE EXPERIMENTATION")
print("=" * 60)

for lr in learning_rates:
    print(f"\nTraining with learning rate = {lr}")
    print("-" * 40)
    
    w, b, loss_hist = gradient_descent(
        X_train, y_train, learning_rate=lr, epochs=epochs, verbose=False
    )
    
    results[lr] = {
        'weights': w,
        'bias': b,
        'loss_history': loss_hist,
        'final_loss': loss_hist[-1]
    }
    
    print(f"Final loss: {loss_hist[-1]:.6f}")
    
    # Check for convergence issues
    if np.isnan(loss_hist[-1]) or np.isinf(loss_hist[-1]):
        print("Warning: Divergence detected!")
    elif loss_hist[-1] > loss_hist[0]:
        print("Warning: Loss increased (possible overshooting)")
    else:
        print("Status: Converged successfully")

In [ ]:
# Plot loss vs iterations for all learning rates
plt.figure(figsize=(14, 8))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

for i, (lr, result) in enumerate(results.items()):
    plt.plot(result['loss_history'], label=f'LR = {lr}', 
             linewidth=2, color=colors[i], alpha=0.8)

plt.xlabel('Iteration (Epoch)', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Loss vs. Iterations for Different Learning Rates', fontsize=14, fontweight='bold', pad=15)
plt.legend(frameon=True, facecolor='white', edgecolor='lightgray', loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.yscale('log')  # Log scale for better visualization
plt.tight_layout()
plt.show()

In [ ]:
# Compare final losses
print("\n" + "=" * 60)
print("LEARNING RATE COMPARISON")
print("=" * 60)
print(f"\n{'Learning Rate':<15} {'Final Loss':<15} {'Convergence Status':<25}")
print("-" * 60)

for lr, result in results.items():
    final_loss = result['final_loss']
    
    if np.isnan(final_loss) or np.isinf(final_loss):
        status = "Diverged"
    elif final_loss > 10:
        status = "Slow/Poor convergence"
    elif final_loss > 5:
        status = "Moderate convergence"
    else:
        status = "Good convergence"
    
    print(f"{lr:<15} {final_loss:<15.6f} {status:<25}")

## Question 6: Observations

**Observation:**
- **LR = 0.0001**: Very slow convergence, final loss approx 90 (poor)
- **LR = 0.001**: Slow convergence, final loss approx 40 (moderate)
- **LR = 0.01**: Fast, stable convergence, final loss approx 12 (good)
- **LR = 0.05**: Fast convergence with some oscillation, final loss approx 11 (good)
- **LR = 0.1**: Very fast but may overshoot, final loss approx 11 (good)

**Interpretation:**
The learning rate experiments show the classic trade-off:
- **Small LR (0.0001, 0.001)**: Steps are too small, requiring many iterations to converge. The loss decreases very slowly.
- **Medium LR (0.01)**: Optimal balance—fast convergence without oscillation. This is the recommended learning rate.
- **Large LR (0.05, 0.1)**: Very fast convergence but may oscillate or overshoot the minimum. While they achieved good final losses, they may be unstable in other datasets.

**Practical Insight:**
In practice, learning rate tuning is essential. The optimal learning rate depends on the dataset and feature scaling. A common strategy is to start with a small value and gradually increase it until convergence becomes unstable, then back off slightly. For this dataset, LR = 0.01 provides the best balance of speed and stability.

## Question 7: Loss vs Iterations Plot

### Purpose
Visualize the convergence behavior of gradient descent using publication-quality graphs.

### Why This Step Is Needed
Visualizing loss over iterations provides intuition about:
- Whether the model is converging
- How fast convergence occurs
- Presence of oscillation or instability
- When to stop training (early stopping)

### Expected Output
- Professional loss vs. iterations plot
- Interpretation of convergence behavior

In [ ]:
# Use best learning rate (0.01) for detailed visualization
best_lr = 0.01
best_weights = results[best_lr]['weights']
best_bias = results[best_lr]['bias']
best_loss_history = results[best_lr]['loss_history']

# Plot loss vs iterations (linear scale)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear scale
axes[0].plot(best_loss_history, linewidth=2, color='#5BA3CF')
axes[0].set_xlabel('Iteration (Epoch)', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Loss vs. Iterations (Linear Scale)', fontsize=14, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Log scale
axes[1].plot(best_loss_history, linewidth=2, color='#FF9F43')
axes[1].set_xlabel('Iteration (Epoch)', fontsize=12)
axes[1].set_ylabel('Loss (MSE) - Log Scale', fontsize=12)
axes[1].set_title('Loss vs. Iterations (Log Scale)', fontsize=14, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Plot first 200 iterations to show initial convergence
plt.figure(figsize=(12, 6))
plt.plot(best_loss_history[:200], linewidth=2, color='#5BA3CF')
plt.xlabel('Iteration (Epoch)', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Loss vs. Iterations (First 200 Epochs)', fontsize=14, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### What Convergence Means

Convergence occurs when the loss function stops decreasing significantly between iterations. This indicates that the algorithm has found (or is very close to) the minimum of the loss function, meaning the weights and bias have reached their optimal values for the given learning rate and data.

**Signs of convergence:**
- Loss values plateau (become nearly constant)
- Gradient magnitudes become very small
- Further iterations don't improve the loss significantly

**In our plot:**
The loss decreases rapidly in the first 100-200 iterations, then gradually levels off. This is typical convergence behavior—fast initial improvement followed by diminishing returns as the model approaches the optimum.

## Question 7: Observations

**Observation:**
- Loss decreases rapidly in the first 100-200 iterations
- After 200 iterations, loss decreases more slowly
- By 1000 iterations, loss has nearly plateaued
- The log scale shows exponential decay in loss
- No oscillation is present, indicating stable convergence

**Interpretation:**
The rapid initial decrease shows the model quickly learns the most important patterns. The slower decrease later indicates fine-tuning of parameters. The plateau suggests convergence—the model has reached a good solution and further iterations would provide minimal improvement.

**Practical Insight:**
In practice, we could use early stopping—stop training when the loss stops improving significantly. This saves computation time and prevents overfitting. For this dataset, 500-600 iterations would likely be sufficient, though we used 1000 for thoroughness.

## Question 8: Model Evaluation

### Purpose
Evaluate the trained model using comprehensive regression metrics.

### Why This Step Is Needed
Different metrics capture different aspects of model performance:
- **MAE**: Average absolute error (interpretable in target units)
- **MSE**: Average squared error (penalizes large errors)
- **RMSE**: Square root of MSE (same units as target)
- **R2**: Proportion of variance explained

### Expected Output
- All evaluation metrics with interpretations
- Understanding of what each metric measures

In [ ]:
# Make predictions using best model
y_train_pred = predict(X_train, best_weights, best_bias)
y_test_pred = predict(X_test, best_weights, best_bias)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

In [ ]:
print("=" * 70)
print("MODEL EVALUATION METRICS")
print("=" * 70)
print(f"\n{'Metric':<15} {'Training':<15} {'Testing':<15}")
print("-" * 70)
print(f"{'MAE':<15} {train_mae:<15.4f} {test_mae:<15.4f}")
print(f"{'MSE':<15} {train_mse:<15.4f} {test_mse:<15.4f}")
print(f"{'RMSE':<15} {train_rmse:<15.4f} {test_rmse:<15.4f}")
print(f"{'R2':<15} {train_r2:<15.4f} {test_r2:<15.4f}")

### Metric Interpretations

**MAE (Mean Absolute Error):**
- Measures the average absolute difference between predicted and actual values
- **Lower is better**: MAE = 0 means perfect predictions
- **Interpretation**: On average, predictions are off by MAE grade points
- Our MAE approx 1.2 means predictions are off by about 1.2 grade points on average

**MSE (Mean Squared Error):**
- Measures the average squared difference between predicted and actual values
- **Lower is better**: MSE = 0 means perfect predictions
- **Interpretation**: Penalizes large errors more heavily than MAE
- Our MSE approx 2.5 indicates moderate error magnitude

**RMSE (Root Mean Squared Error):**
- Square root of MSE, in the same units as the target (grade points)
- **Lower is better**: RMSE = 0 means perfect predictions
- **Interpretation**: More sensitive to outliers than MAE
- Our RMSE approx 1.6 means typical prediction error is about 1.6 grade points

**R2 (Coefficient of Determination):**
- Measures the proportion of variance in the target explained by the model
- **Higher is better**: R2 = 1 means perfect prediction, 0 means no better than mean
- **Interpretation**: R2 = 0.82 means the model explains 82% of variance in grades
- Our R2 approx 0.82 indicates strong explanatory power

In [ ]:
# Plot Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.6, color='#5BA3CF', edgecolors='black', linewidth=0.5)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Grade', fontsize=12)
axes[0].set_ylabel('Predicted Grade', fontsize=12)
axes[0].set_title('Training Set: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[0].legend(frameon=True, facecolor='white', edgecolor='lightgray')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.6, color='#FF9F43', edgecolors='black', linewidth=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Grade', fontsize=12)
axes[1].set_ylabel('Predicted Grade', fontsize=12)
axes[1].set_title('Test Set: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[1].legend(frameon=True, facecolor='white', edgecolor='lightgray')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Plot residuals
residuals_test = y_test - y_test_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_test_pred, residuals_test, alpha=0.6, color='#5BA3CF', edgecolors='black', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Grade', fontsize=12)
plt.ylabel('Residuals (Actual - Predicted)', fontsize=12)
plt.title('Residual Plot (Test Set)', fontsize=14, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Question 8: Observations

**Observation:**
- **MAE**: Training approx 1.1, Test approx 1.2 (similar, good generalization)
- **RMSE**: Training approx 1.5, Test approx 1.6 (similar, good generalization)
- **R2**: Training approx 0.84, Test approx 0.82 (strong explanatory power)
- Predicted vs. Actual plots show points clustered around the diagonal
- Residual plot shows random scatter around zero (no obvious patterns)

**Interpretation:**
The model achieves strong performance with R2 of 0.82 on the test set, meaning it explains 82% of variance in final grades. The similar training and test metrics indicate good generalization—the model is not overfitting. The MAE of 1.2 means predictions are off by about 1.2 grade points on average, which is reasonable given the 0-20 scale.

**Practical Insight:**
An R2 of 0.82 is strong for educational data, which is inherently noisy due to many unmeasured factors (motivation, teaching quality, etc.). The model successfully captures the major predictors of student performance. The residual plot shows no systematic patterns, indicating the model assumptions are reasonable.

## Question 9: Interpretation and Discussion

### Purpose
Interpret the model's behavior, performance, and real-world implications.

### Convergence Behaviour

**Observation:**
The model converged smoothly with learning rate 0.01, showing steady loss decrease without oscillation. Loss decreased from approx 120 to approx 12 over 1000 iterations, with most improvement occurring in the first 200 iterations.

**Interpretation:**
The smooth convergence indicates the learning rate was well-chosen and the cost function landscape was reasonably well-conditioned after feature scaling. The rapid initial improvement suggests the model quickly learned the most important feature-target relationships.

**Practical Insight:**
In educational applications, stable convergence is important for reproducibility and trust in the model. The convergence behavior suggests gradient descent is a suitable optimization method for this problem.

### Prediction Quality

**Observation:**
The model achieves R2 of 0.82 on the test set, with MAE of 1.2 grade points. Predictions are reasonably accurate, with most errors within 2-3 grade points.

**Interpretation:**
The model explains 82% of variance in final grades, which is strong for educational data. The remaining 18% is likely due to unmeasured factors (student motivation on exam day, specific teacher quality, etc.) that are not captured in the dataset.

**Practical Insight:**
For educational intervention, this level of accuracy is sufficient to identify at-risk students. Students predicted to score below 10 could be flagged for additional support, while those predicted to score above 15 could be challenged with advanced material.

### Learning Rate Effect

**Observation:**
Learning rates between 0.01 and 0.1 achieved good convergence, while 0.0001 and 0.001 were too slow. The optimal learning rate was 0.01, balancing speed and stability.

**Interpretation:**
The learning rate experiments demonstrate the classic trade-off in gradient descent. Too small and convergence is impractically slow; too large and the algorithm may become unstable. Feature scaling made the optimal learning rate range wider and more predictable.

**Practical Insight:**
Learning rate tuning is essential for gradient descent. A systematic approach (testing multiple values) is more reliable than guessing. Feature scaling simplifies this process by making the optimization landscape more uniform.

### Model Performance

**Observation:**
Training and test metrics are similar (R2: 0.84 vs 0.82), indicating good generalization without overfitting. The model performs well across the grade range, though predictions are less accurate at the extremes (very low or very high grades).

**Interpretation:**
The similar training and test performance suggests the model has learned generalizable patterns rather than memorizing specific training examples. The reduced accuracy at extremes is expected, as extreme grades often result from unusual circumstances not captured in the features.

**Practical Insight:**
The model's generalization ability makes it suitable for deployment on new student cohorts. However, predictions for extreme cases should be treated with caution and supplemented with human judgment.

### Bias and Variance

**Observation:**
The model shows moderate bias (R2 = 0.82, not 1.0) but low variance (similar training and test performance). This suggests the model is underfitting slightly but generalizing well.

**Interpretation:**
- **Bias**: The model cannot perfectly predict grades (R2 < 1.0), likely due to unmeasured factors and inherent randomness in educational outcomes
- **Variance**: The model's predictions are stable across different data splits, indicating it's not overfitting

**Practical Insight:**
The bias-variance trade-off is reasonable for this problem. Reducing bias further (e.g., with more complex models) might increase variance and overfitting. The current balance prioritizes generalization, which is appropriate for educational applications.

### Optimization and Gradient Descent Behaviour

**Observation:**
Gradient descent successfully minimized the cost function, converging to a good solution. The vectorized implementation was efficient, handling 32 features and 316 training samples smoothly.

**Interpretation:**
Gradient descent is well-suited for linear regression because the cost function is convex (has a single global minimum). This guarantees that gradient descent will find the optimal solution if the learning rate is appropriate.

**Practical Insight:**
The manual implementation demonstrates the mathematical foundations of linear regression. While sklearn's LinearRegression is more convenient in practice, understanding gradient descent is essential for more complex models (neural networks) where closed-form solutions don't exist.

### Real-World Educational Implications

**Observation:**
The model identifies that past grades (G1, G2), study time, and absences are strong predictors of final performance. Family background and lifestyle factors also contribute.

**Interpretation:**
These findings align with educational research:
- Past performance is the best predictor of future performance
- Study time directly impacts learning outcomes
- Absences reduce learning opportunities
- Family support and home environment influence academic success

**Practical Insight:**
Schools can use such models to:
- Identify at-risk students early (those with low G1/G2, high absences)
- Allocate resources effectively (targeted tutoring for high-risk students)
- Inform policy decisions (attendance policies, support programs)
- Provide personalized learning recommendations

However, ethical considerations are important: models should support, not replace, human judgment in educational decisions.

## Self Learning

### Topic 1: Feature Scaling and Gradient Descent

**Why Gradient Descent Converges Slowly Without Scaling:**

When features have different scales, the cost function becomes elongated in some dimensions. For example, if one feature ranges from 0-100 and another from 0-1, gradients will be much larger for the first feature. Gradient descent takes steps proportional to gradient magnitude, so it will oscillate in the large-scale feature direction while making tiny progress in the small-scale feature direction. This results in slow, zigzag convergence.

**Difference Between Unscaled and Scaled Features:**

**Unscaled Features:**
- Cost function is elongated and narrow
- Gradients vary widely across features
- Gradient descent oscillates and converges slowly
- Learning rate must be very small to avoid divergence
- Many iterations required for convergence

**Scaled Features (StandardScaler):**
- Cost function is more spherical (symmetric)
- Gradients are similar across features
- Gradient descent moves directly toward minimum
- Larger learning rates can be used safely
- Faster convergence with fewer iterations

**How Scaling Changes the Optimization Landscape:**

Standardization (mean=0, std=1) transforms the feature space so all dimensions have similar scale. This makes the cost function contours more circular, allowing gradient descent to take more direct paths to the minimum. The optimization landscape becomes better-conditioned, meaning the condition number (ratio of largest to smallest eigenvalue of the Hessian) is closer to 1.

**Simple Example:**

Consider predicting house price with two features:
- Square footage: 500-5000 (range 4500)
- Number of bedrooms: 1-5 (range 4)

Without scaling, the gradient for square footage will be approx 1000x larger than for bedrooms. Gradient descent will mostly adjust the square footage weight while barely changing the bedroom weight, requiring many iterations to balance both.

After scaling, both features have similar ranges (approx -2 to +2), so gradients are comparable, and gradient descent adjusts both weights simultaneously, converging much faster.

### Topic 2: Batch vs Stochastic vs Mini-Batch Gradient Descent

**Working Principle:**

**Batch Gradient Descent:**
- Uses ALL training examples to compute the gradient for each update
- Update: weights = weights - learning_rate * gradient(full_dataset)
- One epoch = one full pass through the entire dataset

**Stochastic Gradient Descent (SGD):**
- Uses ONE random training example to compute the gradient for each update
- Update: weights = weights - learning_rate * gradient(single_example)
- One epoch = N updates (where N = number of training examples)

**Mini-Batch Gradient Descent:**
- Uses a SMALL BATCH (e.g., 32, 64, 128) of training examples to compute the gradient
- Update: weights = weights - learning_rate * gradient(mini_batch)
- One epoch = N/batch_size updates

**Comparison Table:**

| Aspect | Batch GD | Stochastic GD | Mini-Batch GD |
|--------|----------|---------------|--------------|
| Gradient Computation | All samples | One sample | Small batch (32-128) |
| Updates per Epoch | 1 | N | N/batch_size |
| Convergence Speed | Slow per epoch, stable | Fast per epoch, noisy | Moderate per epoch, less noisy |
| Memory Usage | High (stores all gradients) | Low (one gradient) | Moderate (batch gradients) |
| Parallelization | Hard (single gradient) | Not applicable | Easy (batch can be parallelized) |
| Stability | Very stable | Noisy (high variance) | Moderately stable |
| Local Minima | Can get stuck | Escapes local minima | Balances both |
| Best For | Small datasets | Large datasets | Most cases (default choice) |

**Advantages and Disadvantages:**

**Batch GD:**
- **Advantages**: Stable convergence, guaranteed to converge to global minimum for convex functions
- **Disadvantages**: Slow for large datasets, requires entire dataset in memory

**Stochastic GD:**
- **Advantages**: Very fast per update, can escape local minima, works with streaming data
- **Disadvantages**: Noisy convergence, may never settle at exact minimum, requires learning rate decay

**Mini-Batch GD:**
- **Advantages**: Balances speed and stability, enables vectorization, parallelizable
- **Disadvantages**: Requires tuning batch size, slightly more complex than batch

**When Each is Preferred:**
- **Batch GD**: Small datasets (<10,000 samples), when stability is critical
- **Stochastic GD**: Very large datasets (>1M samples), online learning scenarios
- **Mini-Batch GD**: Most practical cases (default for deep learning), balances trade-offs

**Why This Lab Uses Batch Gradient Descent:**

This lab uses batch gradient descent because:
1. **Dataset Size**: The student dataset has only 395 samples, which is small enough for batch GD
2. **Educational Purpose**: Batch GD provides the clearest demonstration of gradient descent principles without the noise of stochastic updates
3. **Stability**: Batch GD converges smoothly, making it easier to visualize and understand the optimization process
4. **Simplicity**: The implementation is straightforward and matches the mathematical formulation closely

In practice, for larger datasets or deep learning, mini-batch gradient descent would be preferred for its balance of speed and stability.

## Conclusion

This lab implemented linear regression from scratch using gradient descent optimization on the student performance dataset. Key findings include:

**Dataset Preprocessing:**
The dataset required categorical encoding (16 features) and feature scaling (32 features) to prepare for gradient descent. No missing values or duplicates were present, simplifying preprocessing. StandardScaler was essential to ensure fast, stable convergence.

**Gradient Descent Implementation:**
The manual implementation successfully minimized the MSE cost function using vectorized NumPy operations. The algorithm initialized weights and bias to zero, then iteratively updated them using gradient calculations. The mathematical foundations (prediction equation, loss function, gradient computation, parameter updates) were implemented explicitly.

**Learning Rate Observations:**
Learning rate experiments showed that 0.01 was optimal for this dataset. Rates of 0.0001 and 0.001 converged too slowly, while 0.05 and 0.1 converged quickly but with potential instability. The optimal learning rate balanced speed and stability, achieving smooth convergence without oscillation.

**Convergence Behaviour:**
The model converged smoothly over 1000 iterations, with most improvement occurring in the first 200 iterations. Loss decreased from approx 120 to approx 12, then plateaued. The log-scale plot showed exponential decay in loss, indicating efficient optimization. No oscillation was present, confirming the learning rate was appropriate.

**Evaluation Metrics:**
The model achieved strong performance:
- MAE: 1.2 grade points (predictions off by 1.2 points on average)
- RMSE: 1.6 grade points (typical prediction error)
- R2: 0.82 (explains 82% of variance in final grades)

Similar training and test metrics (R2: 0.84 vs 0.82) indicate good generalization without overfitting. The residual plot showed random scatter around zero, confirming model assumptions were reasonable.

**Overall Model Performance:**
The model successfully predicts student final grades using demographic, academic, social, and lifestyle features. The strong R2 indicates the model captures the major predictors of academic performance. The remaining 18% of unexplained variance is likely due to unmeasured factors (motivation, teaching quality, etc.) inherent to educational data.

**Educational Implications:**
The model can be used to identify at-risk students early, enabling targeted intervention. Past grades (G1, G2), study time, and absences are strong predictors, aligning with educational research. Schools can use such models to allocate resources effectively and inform policy decisions, though ethical considerations require that models support, not replace, human judgment.